<div align="center">

<img src="https://raw.githubusercontent.com/winstonsmith1897/DantinoX/main/docs/images/dantinox.png" width="150" alt="DantinoX"/>

</div>

# DantinoX · 09 — Environment & Troubleshooting

<div align="center">

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winstonsmith1897/DantinoX/blob/main/docs/notebooks/09_environment_troubleshooting.ipynb) &nbsp;
[![PyPI](https://img.shields.io/pypi/v/dantinox?color=7c3aed)](https://pypi.org/project/dantinox/) &nbsp;
[![GitHub](https://img.shields.io/badge/GitHub-DantinoX-181717?logo=github)](https://github.com/winstonsmith1897/DantinoX)

</div>

*Diagnose the environment problems that actually break JAX libraries on managed runtimes — and fix them in one cell.*

---

**You’ll learn**
- `dx.doctor()` — one-call health check for version skew and GPU visibility
- The three classic failure modes on Colab and what their errors look like
- Checkpoint provenance — `environment.json` and version-skew warnings on `dx.load`
- Memory levers when you hit OOM

**Runtime** — CPU is fine · ~2 min

---

In [ ]:
!pip install -q uv
!uv pip install --system -q -U "dantinox[data,hub,elf,benchmark]" "flax>=0.12,<0.13" "jax[cuda12]"

## The one cell to run first

`dx.doctor()` checks everything that has ever broken a DantinoX session: jax / jaxlib /
CUDA-plugin alignment, flax and optax minimum versions, missing extras, GPU visibility —
and runs a tiny on-device matmul that catches runtime breakage a plain import misses.

It returns a dict (`problems`, `warnings`, `versions`, `gpu`, `ok`), so you can also
gate scripts on it. From a terminal: `dantinox doctor` (exit code 1 on problems).

In [1]:
import dantinox as dx

report = dx.doctor()
report['ok']

  warnings.warn(



  ████                █    █                █   █
  █   █  ███  ████  █████       ████   ███   █ █ 
  █   █ █   █ █   █   █    █    █   █ █   █   █  
  █   █ █  ██ █   █   █    █    █   █ █   █  █ █ 
  ████   ████ █   █   ██   ███  █   █  ███  █   █

  JAX/Flax transformer library  v0.4.7



E0715 11:36:53.169013 2591202 cuda_executor.cc:1206] [0] Failed to allocate device memory: INTERNAL: [0] Failed to allocate 29.62GiB (31803604992 bytes) of device memory: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
E0715 11:36:53.169727 2591202 cuda_executor.cc:1206] [0] Failed to allocate device memory: INTERNAL: [0] Failed to allocate 26.66GiB (28623243264 bytes) of device memory: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
E0715 11:36:53.170389 2591202 cuda_executor.cc:1206] [0] Failed to allocate device memory: INTERNAL: [0] Failed to allocate 23.99GiB (25760917504 bytes) of device memory: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
E0715 11:36:53.171039 2591202 cuda_executor.cc:1206] [0] Failed to allocate device memory: INTERNAL: [0] Failed to allocate 21.59GiB (23184824320 bytes) of device memory: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
E0715 11:36:53.171687 2591202 cuda_executor.cc:1206] [0] Failed to allocate device memory: INTERNAL: [0] Failed to allocate 19.43GiB (20866340864 by

DantinoX doctor
  dantinox             0.4.7
  jax                  0.9.2
  jaxlib               0.9.2
  flax                 0.12.6
  optax                0.2.8
  jax-cuda12-plugin    0.10.0
  jax-cuda12-pjrt      0.10.0
  transformers         4.44.2
  datasets             4.8.5
  devices              cuda:0
  ✗ jax-cuda12-plugin 0.10.0 vs jaxlib 0.9.2 — the CUDA plugin must match jaxlib exactly (PJRT errors otherwise); fix: pip install -U "jax[cuda12]"


False

## The three classic failure modes

| Error you see | What it means | Fix |
|---|---|---|
| `JaxRuntimeError: PJRT_FFI_UserData … size mismatch` / `plugin is likely built with a later version than the framework` | **jax and the CUDA plugin are out of sync** — a partial upgrade updated one but not the other | `pip install -U "jax[cuda12]"` upgrades jax, jaxlib, and the plugin **together**, then restart the session |
| `TypeError: tuple indices must be integers or slices, not ellipsis` (from `flax/nnx/variablelib.py`) | **flax too old** — `nnx.remat` + `Param[...]` crashes on flax < 0.12 (triggered by `gradient_checkpointing=True` or `optimizer='muon'`) | `pip install -U "flax>=0.12,<0.13"`, restart |
| Everything runs but it's slow and `jax.devices()` shows `CpuDevice` | **No GPU visible** — wrong Colab runtime type, or `CUDA_VISIBLE_DEVICES` points at nothing | Colab: *Runtime → Change runtime type → GPU*. Local: check `CUDA_VISIBLE_DEVICES` |

Golden rule on managed runtimes: **upgrade jax only via `"jax[cuda12]"`**, never alone —
and after any upgrade of an already-imported package, **restart the session**.

## Checkpoint provenance

Every run directory saves an `environment.json` with the exact package versions used at
training time. When you `dx.load()` a checkpoint in a *different* environment, you get a
warning per mismatch:

```
WARNING environment skew — flax: trained with 0.12.6, running 0.11.2
```

If a checkpoint misbehaves after an environment change, this is the first thing to look at.

In [2]:
# What gets recorded (same packages dx.doctor() checks):
import importlib.metadata as md

for pkg in ("dantinox", "jax", "jaxlib", "flax", "optax"):
    try:
        print(f"{pkg:10s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:10s} not installed")

dantinox   0.4.7
jax        0.9.2
jaxlib     0.9.2
flax       0.12.6
optax      0.2.8


## Out of memory?

After the first training step DantinoX prints a VRAM line
(`vram 3.2 GB used · peak 8.1/16 GB (51%)`). When peak approaches the limit, in order of
preference:

1. **Lower `batch_size`** (and raise `grad_accum` to keep the effective batch)
2. **`TrainingConfig(gradient_checkpointing=True)`** — recompute activations on backward
3. **`use_bf16=True`** — halves parameter/activation memory on Ampere+ GPUs
4. Reduce `max_context` or `num_blocks`

Also watch the schedule line in the run header: if you see
`⚠ only N optimizer updates`, your corpus/batch/epochs combination is too small to
converge — degenerate, repetitive generations are the symptom.

---

**Recap** — you learned:
- `dx.doctor()` / `dantinox doctor` as the first move on any broken environment
- how to recognise and fix the PJRT-mismatch, old-flax, and no-GPU failure modes
- `environment.json` provenance and version-skew warnings on load
- the four memory levers, in order

**Next →** [10 · Generation Quality](10_generation_quality.ipynb) · [Open in Colab](https://colab.research.google.com/github/winstonsmith1897/DantinoX/blob/main/docs/notebooks/10_generation_quality.ipynb)